> **Version étudiant** — les cellules d'exercice ne rappellent que la consigne : le code est à écrire entièrement par vous-même, sans squelette imposé. Un exemple travaillé sur un cas analogue précède toujours ce type d'exercice. Les cellules repérées par **Question** n'ont pas de correction automatique : exécutez le code fourni, observez, et répondez par écrit. La version corrigée est téléchargeable depuis la page du cours.

# Arbres de décision et réseaux de neurones

**Notebook 7/9 — Introduction à l'apprentissage supervisé**
*L3 MIASHS → Master, Guillaume Metzler, Université Lyon 2*

On change complètement de famille de modèles dans ce notebook. Les arbres de
décision construisent une hypothèse par une succession de règles simples
(« si $x_1 \le s$ alors... sinon... »), sans jamais supposer de forme
particulière pour la frontière de décision. Les réseaux de neurones, eux,
composent des transformations affines et des fonctions d'activation non
linéaires pour apprendre, couche après couche, une nouvelle représentation
des données. Les deux familles sont utilisables aussi bien en classification
qu'en régression, et toutes deux se règlent par un ou plusieurs
hyperparamètres qui contrôlent leur capacité — d'où un fil conducteur commun
tout au long du notebook : que se passe-t-il quand on augmente cette
capacité ?

Ce notebook couvre :
- les critères de construction d'un arbre (impureté de Gini, entropie, MSE
  en régression) et l'effet de `max_depth` / `min_samples_leaf` ;
- le perceptron, les fonctions d'activation usuelles et le problème XOR ;
- le perceptron multicouche (`MLPClassifier` / `MLPRegressor`), l'effet de
  son architecture et du taux d'apprentissage sur l'entraînement.


## 1. Arbres de décision de classification

Un arbre de décision applique successivement des règles binaires du type
$x_j \le s$ à un jeu de données, ce qui le sépare progressivement en groupes
de plus en plus homogènes. On se limite ici aux **arbres binaires** : chaque
règle sépare un nœud en exactement deux nœuds fils. Selon l'espace de sortie
$\mathcal{Y}$, on parle d'**arbre de classification** ($\mathcal{Y}$ fini) ou
d'**arbre de régression** ($\mathcal{Y} \subset \mathbb{R}$) ; une feuille
prédit alors la classe majoritaire, respectivement la moyenne des $y_i$
qu'elle contient.

Reprenons l'exemple du cours : un jeu de données jouet de 7 personnes (4
femmes, 3 hommes), décrites par leur âge et leur taille. La règle
$\text{Âge} \le 25$ sépare ce nœud en un groupe de gauche **pur** (2 femmes
seulement) et un groupe de droite qui contient encore 2 femmes et 3 hommes.
Pour choisir automatiquement une telle règle, il faut une mesure de la
qualité d'un nœud — son **impureté** — puis une mesure de l'amélioration
apportée par un split, le **gain**.

En classification binaire, avec $p_1$ la proportion d'exemples de la classe
1 dans un nœud $N$, l'**impureté de Gini** et l'**entropie** s'écrivent :

$$G_N = 2\, p_1 (1 - p_1), \qquad \mathrm{Ent}_N = -\sum_{j} p_j \log_2(p_j).$$

$G_N$ vaut 0 pour un nœud pur et 0.5 pour un nœud à parité parfaite. Le
**gain** d'un split qui sépare un nœud en $N_L$ et $N_R$ vaut :

$$\Gamma = Q_{\text{root}} - \left(\frac{|N_L|}{|N_L|+|N_R|} Q_{N_L} + \frac{|N_R|}{|N_L|+|N_R|} Q_{N_R}\right),$$

où $Q$ désigne l'une de ces mesures d'impureté. La concavité de $G$ (et de
l'entropie) garantit, par l'inégalité de Jensen, que $\Gamma \ge 0$ : un
split ne peut jamais dégrader la pureté moyenne des nœuds fils.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p1 = np.linspace(0.001, 0.999, 200)
gini = 2 * p1 * (1 - p1)


def entropie(p):
    q = 1 - p
    t1 = 0.0 if p == 0 else -p * np.log2(p)
    t2 = 0.0 if q == 0 else -q * np.log2(q)
    return t1 + t2


# Noeud racine : 4 femmes, 3 hommes -> p1 = 4/7
p1_root = 4 / 7
G_root = 2 * p1_root * (1 - p1_root)
Ent_root = entropie(p1_root)

# Apres le split "Age <= 25" : gauche pur (2 femmes), droite 2F + 3M
n_L, n_R = 2, 5
G_L = 0.0
p1_R = 2 / 5
G_R = 2 * p1_R * (1 - p1_R)
G_weighted = (n_L / (n_L + n_R)) * G_L + (n_R / (n_L + n_R)) * G_R
gain = G_root - G_weighted

plt.figure(figsize=(7, 4.5))
plt.plot(p1, gini, label=r"$G(p_1) = 2\,p_1(1-p_1)$")
plt.scatter([p1_root], [G_root], color="tab:red", zorder=5, label="nœud racine")
plt.scatter([p1_R], [G_R], color="tab:green", zorder=5, label="nœud droit après split")
plt.hlines(G_weighted, 0, 1, colors="gray", linestyles="dotted", label="moyenne pondérée des fils")
plt.annotate("", xy=(0.05, G_weighted), xytext=(0.05, G_root),
             arrowprops=dict(arrowstyle="<->", color="black"))
plt.text(0.07, (G_root + G_weighted) / 2, f"gain $\\Gamma \\approx {gain:.3f}$")
plt.xlabel(r"proportion $p_1$ de la classe 1")
plt.ylabel("impureté de Gini")
plt.legend(loc="lower center")
plt.tight_layout()
plt.show()

print(f"G_root = {G_root:.4f}   Ent_root = {Ent_root:.4f}")
print(f"G_droit après split = {G_R:.4f} (nœud gauche pur, G = 0)")
print(f"Gain de Gini Gamma = {gain:.4f}")

# On verifie la formule fermee 2p(1-p) contre une somme explicite sur les 2 classes
G_root_somme = p1_root * (1 - p1_root) + (1 - p1_root) * p1_root
assert abs(G_root - G_root_somme) < 1e-12


$$ $$

**Question :** Que signifie une impureté de Gini nulle pour un nœud ? Et pourquoi le gain $\Gamma$ obtenu ici est-il forcément positif, quelle que soit la règle de split choisie ?

$$ $$

### Exercice 1 : impureté de Gini et entropie d'un autre nœud

On considère cette fois un nœud contenant 17 exemples, dont 12 de la classe
« positif » et 5 de la classe « négatif ».

In [ ]:
# Calculez la proportion p d'exemples positifs, l'impureté de Gini
# G = 2 p (1-p) et l'entropie Ent = -(p log2 p + (1-p) log2(1-p)) de ce nœud.
# Affichez les trois valeurs, et vérifiez par un assert que votre calcul
# d'entropie via une fonction dédiée coïncide avec un calcul terme à terme
# direct.


### 1.1 Profondeur de l'arbre et sur-apprentissage

En répétant le principe précédent jusqu'à obtenir des feuilles pures, on
construit toujours un arbre qui classe parfaitement les données
d'apprentissage — mais un tel arbre **sur-apprend** en général et généralise
mal. La **profondeur maximale** (`max_depth`) contrôle directement cette
complexité : un arbre trop peu profond (`max_depth=1`, un simple *stump*)
**sous-apprend** ; un arbre sans limite de profondeur isole parfois chaque
exemple bruité dans sa propre feuille.

Regardons l'effet de `max_depth` sur la frontière de décision et
l'exactitude, sur un jeu de données non linéairement séparable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

depths = [1, 3, 5, None]
fig, axes = plt.subplots(1, len(depths), figsize=(18, 4.5))

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                      np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))

for ax, depth in zip(axes, depths):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    clf.fit(X_train, y_train)
    acc_train = clf.score(X_train, y_train)
    acc_test = clf.score(X_test, y_test)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=15)
    label_depth = "aucune limite" if depth is None else str(depth)
    ax.set_title(f"max_depth={label_depth}\ntrain={acc_train:.2f}  test={acc_test:.2f}")

fig.suptitle("Frontière de décision d'un arbre de décision selon sa profondeur (make_moons)")
plt.tight_layout()
plt.show()


$$ $$

**Question :** Que devient l'exactitude sur le train et sur le test quand `max_depth` augmente ? Quelle profondeur retiendriez-vous ici ?

$$ $$

On peut afficher directement la structure d'un arbre entraîné (variable
et seuil testés, impureté, nombre d'exemples par nœud) avec
`sklearn.tree.plot_tree`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)

clf = DecisionTreeClassifier(max_depth=3, random_state=0)
clf.fit(X, y)

plt.figure(figsize=(13, 7))
plot_tree(clf, feature_names=["x1", "x2"], class_names=["classe 0", "classe 1"],
          filled=True, rounded=True, fontsize=9)
plt.title("Structure d'un arbre de décision (max_depth=3, make_moons)")
plt.show()


### Exercice 2 : structure et importance des variables sur `load_wine`

`load_wine` décrit 178 vins par 13 mesures physico-chimiques, répartis en 3
cépages. Une fois un arbre entraîné, on peut mesurer l'**importance** de
chaque variable : la réduction totale d'impureté qu'elle apporte, pondérée
par le nombre d'exemples concernés et cumulée sur tout l'arbre.

In [ ]:
# Chargez load_wine, entraînez un DecisionTreeClassifier(max_depth=4,
# random_state=0), affichez sa structure avec plot_tree, puis affichez un
# graphique en barres de clf.feature_importances_ (triées par ordre
# décroissant, avec les noms des variables). Indiquez, dans un print, la
# variable la plus importante.


### 1.2 Taille minimale des feuilles

`min_samples_leaf` impose un nombre minimal d'exemples dans chaque feuille :
plus cette valeur est grande, plus l'arbre est contraint à des feuilles
« généralistes », ce qui limite le sur-apprentissage d'une autre manière que
`max_depth`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.tree import DecisionTreeClassifier

X, y = make_circles(n_samples=300, noise=0.15, factor=0.4, random_state=0)

leaf_values = [1, 5, 25]
fig, axes = plt.subplots(1, len(leaf_values), figsize=(15, 5))

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 300),
                      np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 300))

for ax, leaf in zip(axes, leaf_values):
    clf = DecisionTreeClassifier(max_depth=8, min_samples_leaf=leaf, random_state=0)
    clf.fit(X, y)
    acc = clf.score(X, y)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(f"min_samples_leaf={leaf}\ntrain={acc:.2f}")

fig.suptitle("Effet de min_samples_leaf sur la frontière (max_depth=8 fixé, make_circles)")
plt.tight_layout()
plt.show()


$$ $$

**Question :** Quel est l'effet visuel de `min_samples_leaf` sur la frontière de décision ? Pourquoi une valeur trop faible favorise-t-elle le sur-apprentissage même avec `max_depth` fixé ?

$$ $$

### 1.3 Choisir les hyperparamètres par validation croisée

Comme pour tout modèle, on choisit `max_depth` (et les autres paramètres
d'élagage) par validation croisée plutôt qu'à l'œil, avec
`sklearn.model_selection.validation_curve`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

iris = load_iris()
X, y = iris.data, iris.target

depths = np.arange(1, 9)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=0), X, y,
    param_name="max_depth", param_range=depths, cv=5, scoring="accuracy",
)

plt.figure(figsize=(7, 4.5))
plt.plot(depths, train_scores.mean(axis=1), marker="o", label="exactitude (apprentissage)")
plt.plot(depths, val_scores.mean(axis=1), marker="o", label="exactitude (validation croisée)")
plt.xlabel("max_depth")
plt.ylabel("exactitude moyenne")
plt.title("Courbe de validation : max_depth (DecisionTreeClassifier, load_iris)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Meilleure profondeur (validation) : {depths[np.argmax(val_scores.mean(axis=1))]}")


### Exercice 3 (Master) : réglage conjoint de `max_depth` et `min_samples_leaf`

On travaille sur `load_breast_cancer` (30 variables, 2 classes).

In [ ]:
# Séparez load_breast_cancer en train/test (70/30, random_state=0,
# stratify=y). Tracez une courbe de validation (5 plis) pour max_depth (1 à
# 12) puis pour min_samples_leaf (1 à 30), à l'aide de validation_curve.
# Utilisez ensuite GridSearchCV pour trouver la meilleure combinaison des
# deux hyperparamètres sur le train, et évaluez le modèle retenu sur le
# test.


## 2. Arbres de régression

Pour un arbre de régression, le critère d'impureté d'un nœud $N$ n'est plus
Gini ou l'entropie mais l'**erreur quadratique moyenne (MSE)** entre les
$y_i$ du nœud et leur moyenne $\bar y$ :

$$\mathrm{MSE}_N = \frac{1}{m_N} \sum_{i=1}^{m_N} (y_i - \bar y)^2.$$

C'est exactement la variance des $y_i$ dans le nœud. Le gain se définit de
la même façon que pour la classification, en remplaçant $Q$ par $\mathrm{MSE}$ :
on choisit à chaque étape la variable et le seuil qui réduisent le plus
cette erreur. Une feuille prédit alors la **moyenne** des $y_i$ qu'elle
contient : la fonction apprise par un arbre de régression est nécessairement
**en escalier**, constante par morceaux.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.RandomState(0)
X = np.sort(10 * rng.rand(200, 1), axis=0)
y = (X * np.sin(X)).ravel() + rng.normal(0, 1.0, X.shape[0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
X_plot = np.linspace(0, 10, 500).reshape(-1, 1)

depths = [2, 5, None]
fig, axes = plt.subplots(1, len(depths), figsize=(16, 4.5), sharey=True)

for ax, depth in zip(axes, depths):
    reg = DecisionTreeRegressor(max_depth=depth, random_state=0)
    reg.fit(X_train, y_train)
    mse_train = mean_squared_error(y_train, reg.predict(X_train))
    mse_test = mean_squared_error(y_test, reg.predict(X_test))

    ax.scatter(X_train, y_train, s=12, alpha=0.5, label="train")
    ax.scatter(X_test, y_test, s=12, alpha=0.5, marker="s", label="test")
    ax.plot(X_plot, reg.predict(X_plot), color="black", linewidth=2)
    label_depth = "aucune limite" if depth is None else str(depth)
    ax.set_title(f"max_depth={label_depth}\nMSE train={mse_train:.2f}  MSE test={mse_test:.2f}")
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[0].legend()
fig.suptitle("Arbre de régression : prédiction en escalier selon la profondeur")
plt.tight_layout()
plt.show()


$$ $$

**Question :** Pourquoi les prédictions d'un arbre de régression forment-elles toujours une fonction en escalier ? Comment évoluent la MSE d'apprentissage et de test avec `max_depth` ?

$$ $$

### Exercice 4 (Master) : arbre de régression sur `load_diabetes`

`load_diabetes` contient 10 variables cliniques et une mesure quantitative
de la progression du diabète un an après le diagnostic (la cible).

In [ ]:
# Chargez load_diabetes, séparez en train/test (70/30, random_state=0).
# À l'aide de validation_curve (scoring='neg_mean_squared_error', cv=5),
# tracez la MSE moyenne en apprentissage et en validation pour max_depth
# de 1 à 15. Entraînez le modèle final avec la profondeur retenue et
# affichez son RMSE et son R2 sur le test ; comparez à un arbre
# max_depth=None (comparaison du sur-apprentissage).


## 3. Le perceptron

Le premier modèle mathématique de neurone (McCulloch et Pitts, 1943),
formalisé plus tard sous le nom de **perceptron** (Rosenblatt, 1958), prend
en entrée un vecteur $x \in \mathbb{R}^d$ et dépend d'un paramètre
$(w, b) \in \mathbb{R}^{d+1}$. Sa sortie s'écrit :

$$h(x) = f\big(\langle w, x\rangle + b\big),$$

où $f$ est une **fonction d'activation**. Historiquement, $f$ est la
fonction de **Heavyside** ($f(z)=0$ si $z<0$, $f(z)=1$ sinon) — exactement
la même règle de décision qu'un SVM linéaire. La **règle de Hebb** met à
jour $w$ et $b$ uniquement à partir des exemples mal classés $i \in I$ :

$$w \leftarrow w + \alpha\, y_i x_i, \qquad b \leftarrow b + \alpha\, y_i,$$

où $\alpha$ est le **taux d'apprentissage**. La **règle de Widrow-Hoff**
(loi du delta) prend en plus en compte l'erreur commise :
$w \leftarrow w + (y_i - h(x_i))\, x_i$. En pratique, on remplace souvent
Heavyside par une version lisse, mieux adaptée à la descente de gradient :
**sigmoïde** $\sigma(x) = 1/(1+e^{-x})$, **tangente hyperbolique**
$\tanh(x)$, ou **ReLU** $\max(0, x)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 400)
heavyside = (x >= 0).astype(float)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)

plt.figure(figsize=(8, 5))
plt.plot(x, heavyside, label="Heavyside", linestyle="--")
plt.plot(x, sigmoid, label=r"sigmoïde $\sigma(x)$")
plt.plot(x, tanh, label=r"$\tanh(x)$")
plt.plot(x, relu, label=r"ReLU$(x)=\max(0,x)$")
plt.axhline(0, color="gray", linewidth=0.6)
plt.axvline(0, color="gray", linewidth=0.6)
plt.legend()
plt.title("Fonctions d'activation usuelles")
plt.tight_layout()
plt.show()

# Sortie d'un perceptron de parametres w, b pour une entree x
w = np.array([0.5, -1.0, 2.0])
b = -0.2
x_in = np.array([2.0, 1.0, 0.0])

z = np.dot(w, x_in) + b
h_heavyside = 1.0 if z >= 0 else 0.0
h_sigmoid = 1 / (1 + np.exp(-z))
h_tanh = np.tanh(z)

print(f"z = <w,x> + b = {z:.4f}")
print(f"sortie Heavyside : {h_heavyside}")
print(f"sortie sigmoïde   : {h_sigmoid:.4f}")
print(f"sortie tanh       : {h_tanh:.4f}")

assert abs(z - (-0.2)) < 1e-9


### Exercice 5 : sortie d'un perceptron, autre configuration

On considère à présent un perceptron de paramètres $w = (1, 1, 1)$ et
$b = -1$.

In [ ]:
# Pour l'entrée x = (2, 0, 1), calculez z = <w,x> + b, puis la sortie
# obtenue avec Heavyside, avec la sigmoïde et avec tanh. Comparez les trois
# valeurs par un print, et vérifiez z par un assert.


### 3.1 Le problème XOR

Un perceptron simple sépare parfaitement les jeux de données **OR** et
**AND** (linéairement séparables), mais échoue sur **XOR** ("ou exclusif") :
aucune droite ne peut séparer les deux classes. Construisons ce jeu de
données et vérifions qu'un `Perceptron` linéaire de `scikit-learn` échoue
bien à le résoudre.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Perceptron

rng = np.random.RandomState(1)
n_par_coin = 60
centres = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
labels_coins = np.array([0, 1, 1, 0])  # XOR(x1, x2)

X_parts, y_parts = [], []
for c, lab in zip(centres, labels_coins):
    X_parts.append(c + 0.12 * rng.randn(n_par_coin, 2))
    y_parts.append(np.full(n_par_coin, lab))
X_xor = np.vstack(X_parts)
y_xor = np.concatenate(y_parts)

clf_lin = Perceptron(max_iter=1000, random_state=0)
clf_lin.fit(X_xor, y_xor)
acc_lin = clf_lin.score(X_xor, y_xor)

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
Z = clf_lin.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(5.5, 5.5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", edgecolor="k", s=25)
plt.title(f"Problème XOR — Perceptron linéaire (exactitude = {acc_lin:.2f})")
plt.tight_layout()
plt.show()

print(f"Exactitude du perceptron linéaire sur XOR : {acc_lin:.2f}")


$$ $$

**Question :** Pourquoi un perceptron simple (modèle linéaire) ne peut-il pas résoudre le problème XOR, alors qu'il résout OR et AND sans difficulté ?

$$ $$

### Exercice 6 : `Perceptron` contre `MLPClassifier` sur `make_circles`

`make_circles` produit un autre jeu de données non linéairement séparable
(deux cercles concentriques), sur le même principe que XOR.

In [ ]:
# Générez make_circles(n_samples=300, noise=0.1, factor=0.4,
# random_state=0). Entraînez un Perceptron linéaire et un
# MLPClassifier(hidden_layer_sizes=(8,), activation='tanh', max_iter=3000,
# random_state=0) sur ces données, affichez les deux exactitudes, et tracez
# la frontière de décision du MLP.


## 4. Perceptron multicouche

Composer plusieurs perceptrons en couches successives donne un **réseau de
neurones multicouche** (*Multi-Layer Perceptron*, MLP) :

- la **couche d'entrée**, de taille égale à la dimension de l'entrée (plus
  un neurone de biais) ;
- une ou plusieurs **couches cachées**, dont le nombre et la taille sont
  choisis par l'utilisateur ;
- la **couche de sortie**, dont la taille dépend du problème (1 neurone en
  régression ou classification binaire, $K$ neurones pour $K$ classes).

Un réseau est **entièrement connecté** (*fully connected*) lorsque tous les
neurones d'une couche sont reliés à tous ceux de la couche suivante. Pour un
réseau à $K$ couches cachées de dimensions $d^{(1)}, \dots, d^{(K)}$, entre
une entrée de dimension $d^{(0)}$ et une sortie de dimension $d^{(K+1)}$, le
nombre total de paramètres (poids + biais) vaut
$\sum_{k=0}^{K} d^{(k+1)} (d^{(k)}+1)$.

Regardons comment l'architecture (`hidden_layer_sizes`) influence la
flexibilité de la frontière apprise, sur le même jeu de données `make_moons`
que pour les arbres — de façon à pouvoir comparer directement les deux
familles de modèles plus loin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
X = StandardScaler().fit_transform(X)

architectures = [(2,), (10,), (20, 10)]
fig, axes = plt.subplots(1, len(architectures), figsize=(16, 4.5))

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                      np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))

for ax, arch in zip(axes, architectures):
    clf = MLPClassifier(hidden_layer_sizes=arch, activation="tanh", max_iter=3000, random_state=0)
    clf.fit(X, y)
    acc = clf.score(X, y)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(f"hidden_layer_sizes={arch}\ntrain={acc:.2f}")

fig.suptitle("Frontière de décision d'un MLP selon son architecture (make_moons)")
plt.tight_layout()
plt.show()


$$ $$

**Question :** Quel est l'effet du nombre de neurones et de couches cachées sur la frontière de décision ? Faites le parallèle avec l'effet de `max_depth` observé pour les arbres.

$$ $$

### 4.1 Apprentissage : *forward*, *backward*, rétropropagation

Les paramètres d'un réseau sont appris par descente de gradient, en deux
étapes répétées à chaque exemple (ou mini-lot) : le **forward** calcule,
couche après couche, la sortie du réseau et la valeur de la perte ; le
**backward** (**rétropropagation**, *back-propagation*) met à jour tous les
paramètres, y compris ceux des premières couches, en propageant l'erreur de
la sortie vers l'entrée grâce à la **règle de la chaîne** :

$$\frac{\partial (f \circ g)}{\partial x}(x) = \frac{\partial f}{\partial g}(g(x)) \times \frac{\partial g}{\partial x}(x).$$

Pour un réseau très simple à deux couches cachées d'une seule unité chacune,
de sorties intermédiaires $y_1 = f_1(x, w_1)$, $y_2 = f_2(y_1, w_2)$,
$y_3 = f_3(y_2, w_3)$, la dérivée de la sortie par rapport au tout premier
paramètre $w_1$ s'obtient par composition :

$$\frac{\partial y_3}{\partial w_1} = \frac{\partial f_3}{\partial y_2} \times \frac{\partial f_2}{\partial y_1} \times \frac{\partial f_1}{\partial w_1}.$$

C'est cette réutilisation en cascade des dérivées, couche par couche, qui
rend l'entraînement d'un réseau profond réalisable en pratique — sans elle,
il faudrait recalculer entièrement le gradient de chaque paramètre.

### 4.2 Taux d'apprentissage et convergence

Comme pour toute descente de gradient, le **taux d'apprentissage**
(`learning_rate_init` dans `MLPClassifier`) contrôle la taille des pas
effectués à chaque mise à jour. `MLPClassifier` expose `loss_curve_`, la
valeur de la perte à chaque itération, ce qui permet d'observer directement
la convergence.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)

learning_rates = [0.001, 0.05, 2.0]
plt.figure(figsize=(8, 5))
for lr in learning_rates:
    clf = MLPClassifier(hidden_layer_sizes=(10,), solver="sgd", learning_rate_init=lr,
                         max_iter=300, random_state=0)
    clf.fit(X, y)
    plt.plot(clf.loss_curve_, label=f"learning_rate_init={lr}")

plt.xlabel("itération")
plt.ylabel("perte (loss)")
plt.title("Convergence d'un MLP selon le taux d'apprentissage (solver='sgd')")
plt.legend()
plt.tight_layout()
plt.show()


$$ $$

**Question :** Comparez les trois courbes de perte : que se passe-t-il pour un taux d'apprentissage trop petit, et pour un taux trop grand ?

$$ $$

### Exercice 7 (Master) : effet du taux d'apprentissage sur `load_wine`

On reprend le même principe sur un jeu de données réel.

In [ ]:
# Chargez load_wine, standardisez X (StandardScaler). Entraînez un
# MLPClassifier(hidden_layer_sizes=(15,), solver='sgd', max_iter=500,
# random_state=0) pour plusieurs valeurs de learning_rate_init (par exemple
# 0.0005, 0.01, 1.0), tracez les courbes loss_curve_ sur un même graphique,
# et indiquez dans un print la valeur qui atteint la perte finale la plus
# basse.


### 4.3 Arbre de décision contre MLP, sur les mêmes données

Comparons pour finir un `DecisionTreeClassifier` et un `MLPClassifier` sur
le même jeu `make_moons` que celui utilisé tout au long des sections 1 et
4 : mêmes données, deux façons radicalement différentes d'apprendre une
frontière non linéaire.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
X = StandardScaler().fit_transform(X)  # utile pour le MLP, sans effet sur l'arbre
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

modeles = {
    "Arbre de décision (max_depth=5)": DecisionTreeClassifier(max_depth=5, random_state=0),
    "MLPClassifier (20, 10)": MLPClassifier(hidden_layer_sizes=(20, 10), activation="tanh",
                                             max_iter=3000, random_state=0),
}

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                      np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (nom, modele) in zip(axes, modeles.items()):
    modele.fit(X_train, y_train)
    acc_train, acc_test = modele.score(X_train, y_train), modele.score(X_test, y_test)

    Z = modele.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(f"{nom}\ntrain={acc_train:.2f}  test={acc_test:.2f}")

plt.tight_layout()
plt.show()


### Exercice 8 (Master, avancé) : `MLPClassifier` sur `load_digits`, comparaison à un arbre

`load_digits` contient des images 8x8 de chiffres manuscrits (10 classes,
64 variables).

In [ ]:
# Chargez load_digits, standardisez X, séparez en train/test (70/30,
# random_state=0). Entraînez un MLPClassifier pour plusieurs architectures
# (par exemple (10,), (50,), (100,), (50, 25), max_iter=500,
# random_state=0) et affichez les exactitudes train/test pour chacune.
# Entraînez aussi un DecisionTreeClassifier(max_depth=10, random_state=0)
# comme référence, et commentez dans un print la comparaison des deux
# familles de modèles sur ce jeu de données.


## Pour la suite

Les arbres de décision présentés ici sont la brique de base des **méthodes
ensemblistes** (forêts aléatoires, boosting) : combiner de nombreux arbres,
plutôt qu'un seul, permet en général d'améliorer sensiblement la robustesse
et les performances — c'est l'objet du notebook suivant. Les réseaux de
neurones, eux, se généralisent en empilant beaucoup plus de couches et en
utilisant des architectures spécialisées (convolutions pour les images,
couches récurrentes ou mécanismes d'attention pour les séquences) : c'est
tout le champ de l'**apprentissage profond**.